# 01 - Ingest the tennis data
**What it does:** <in your own words, one or two sentences>
**Needs:** notebook 00 has run (the bucket exists)
**Produces:** one CSV per season in the bucket under raw/, and a manifest in manifests/
**Run after:** 00_gcp_setup

## Install the pinned libraries

In [ ]:
# The repo is public, so requirements.txt can be read straight from GitHub.
REQUIREMENTS_URL = "https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

## Sign in and load settings from Colab Secrets
Switch on Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Download from the pinned commit and upload to the bucket
The commit ID pins the exact snapshot of the mirror, so everyone downloads the same files.
Files already in the bucket are skipped, so raw data is never overwritten.

In [ ]:
import hashlib
import json
import requests
from datetime import datetime, timezone
from google.cloud import storage

COMMIT = "83733587353df8a41f2fd4f516147d5aa83f5a8d"   # pins the exact snapshot
assert len(COMMIT) == 40, "Use the full 40-character commit ID"
YEARS = range(2010, 2027)                             # 2010 through 2026

BASE_URL = f"https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/{COMMIT}/wta"
FOLDER = f"raw/tennis_sackmann_archive/{COMMIT[:8]}/wta"

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
manifest = []

for year in YEARS:
    name = f"wta_matches_{year}.csv"
    url = f"{BASE_URL}/{name}"

    # 1. Download
    response = requests.get(url, timeout=60)
    if response.status_code == 404:
        print(f"{name}: not found")
        continue
    response.raise_for_status()
    data = response.content

    # 2. Upload (skip if it is already there, so raw data is never overwritten)
    blob = bucket.blob(f"{FOLDER}/{name}")
    if blob.exists():
        status = "already stored"
    else:
        blob.upload_from_string(data, content_type="text/csv")
        status = "uploaded"

    # 3. Record what we got
    sha256 = hashlib.sha256(data).hexdigest()
    rows = data.count(b"\n") - 1
    manifest.append({"file": name, "url": url, "rows": rows, "sha256": sha256})
    print(f"{name}: {status} | {rows:,} rows | {sha256[:12]}")

# 4. Save the manifest (a timestamped record of this run)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/wta_matches_{COMMIT[:8]}_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"commit": COMMIT, "files": manifest}, indent=2),
    content_type="application/json",
)
print("Manifest saved:", manifest_path)

## Check: are the stored files identical to what was downloaded?

In [ ]:
problems = 0
for entry in manifest:
    stored = bucket.blob(f"{FOLDER}/{entry['file']}").download_as_bytes()
    ok = hashlib.sha256(stored).hexdigest() == entry["sha256"]
    problems += (not ok)
    print("OK  " if ok else "FAIL", entry["file"])
print("files checked:", len(manifest), "| problems:", problems)